In [1]:
# %pip install torch pytorch-lightning transformers peft datasets bitsandbytes

In [4]:
import os


# Clean TensorFlow spam
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"


from torch.utils.data import Dataset, DataLoader
import torch
import pytorch_lightning as pl
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

import torch.utils.data as data
from datasets import load_dataset

In [2]:
model_path = os.path.join(os.path.dirname(os.getcwd()),"Models","Llama3.2-1B-Instruct-hf")
# Verify the path exists
if not os.path.exists(model_path):
    raise FileNotFoundError(f"The directory {model_path} does not exist. Please check the path.")

model_path

'/home/miguel/Desktop/research/Models/Llama3.2-1B-Instruct-hf'

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
# Set pad token (before using the tokenizer)
tokenizer.pad_token = tokenizer.eos_token

special_tokens = {"additional_special_tokens": ["<|user|>", "<|assistant|>"]}
tokenizer.add_special_tokens(special_tokens)
# model.resize_token_embeddings(len(tokenizer))

2

In [4]:
datasetName = "FreedomIntelligence/medical-o1-reasoning-SFT"

In [5]:
# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset(datasetName, "en")
train_data = ds['train']

In [18]:
class process_data():
    def __init__(self,  tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.inputs = []
        self.max_length = max_length

    def __call__(self, data):
        self.convert_information(data)
        self.save_file("medical-o1-reasoning-SFT")

    def create_tokens_text(self, data_text):
    
        conversation = (
            f"<|user|>\n{data_text['Question']}\n"
            f"<|assistant|>\n{data_text['Complex_CoT']}\n\n"
            f"My response is:\n{data_text['Response']}"
        )

        # Tokenize
        encodings = self.tokenizer(conversation, 
                        truncation=True,
                        max_length=self.max_length,
                        padding="max_length",
                        return_tensors="pt"
                    )
    
        # Create labels (for causal language modeling)
        input_ids = encodings["input_ids"][0]
        attention_mask = encodings["attention_mask"][0]
        labels = input_ids.clone()

        # Mask labels for user prompts (optional)
        # This means we only calculate loss on assistant responses
        # Find positions of <|assistant|> tokens
        assistant_positions = []
        assistant_token_id = tokenizer.convert_tokens_to_ids("<|assistant|>")
        for i, token_id in enumerate(input_ids):
            if token_id == assistant_token_id:
                assistant_positions.append(i)

        # Set labels for non-assistant text to -100 (ignored in loss calculation)
        if assistant_positions:
            is_assistant = False
            for i in range(len(labels)):
                if i in assistant_positions:
                    is_assistant = True
                if not is_assistant:
                    labels[i] = -100

        return {
            "text":conversation,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
            }
    

    def convert_information(self, data):
        from tqdm import tqdm

        self.inputs = []
        for item in tqdm(data, desc="Processing items"):
            self.inputs.append(self.create_tokens_text(item))

    def save_file(self, filename):
        import pickle
        with open(f"{filename}.pkl", "wb") as f:
            pickle.dump(self.inputs, f)


In [19]:
dp = process_data(tokenizer=tokenizer, max_length=1024)
dp(train_data)

Processing items: 100%|██████████| 19704/19704 [04:47<00:00, 68.63it/s]


In [ ]:
# class LlamaDataset(Dataset):
#     def __init__(self, data, tokenizer, max_length=512):
#         self.tokenizer = tokenizer
#         self.inputs = []
#         self.max_length = max_length
        
#         for item in data:
#             # Format messages into a single string
#             conversation = (
#                 f"<|user|>\n{item['Question']}\n"
#                 f"<|assistant|>\n{item['Complex_CoT']}\n\n"
#                 f"My response is:\n{item['Response']}"
#             )
            
#             # Tokenize
#             encodings = tokenizer(conversation, 
#                                   truncation=True,
#                                   max_length=self.max_length,
#                                   padding="max_length",
#                                   return_tensors="pt")
            
#             # Create labels (for causal language modeling)
#             input_ids = encodings["input_ids"][0]
#             attention_mask = encodings["attention_mask"][0]
#             labels = input_ids.clone()
            
#             # Mask labels for user prompts (optional)
#             # This means we only calculate loss on assistant responses
#             # Find positions of <|assistant|> tokens
#             assistant_positions = []
#             assistant_token_id = tokenizer.convert_tokens_to_ids("<|assistant|>")
#             for i, token_id in enumerate(input_ids):
#                 if token_id == assistant_token_id:
#                     assistant_positions.append(i)
            
#             # Set labels for non-assistant text to -100 (ignored in loss calculation)
#             if assistant_positions:
#                 is_assistant = False
#                 for i in range(len(labels)):
#                     if i in assistant_positions:
#                         is_assistant = True
#                     if not is_assistant:
#                         labels[i] = -100
            
#             self.inputs.append({
#                 "input_ids": input_ids,
#                 "attention_mask": attention_mask,
#                 "labels": labels
#             })

#     def __len__(self):
#         return len(self.inputs)

#     def __getitem__(self, idx):
#         return self.inputs[idx]

In [8]:
dataR = []
for i, d in enumerate(train_data):
    dataR.append(d)
    if i == 10:
        break

In [9]:
dataR

[{'Question': 'Given the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?',
  'Complex_CoT': "Okay, let's see what's going on here. We've got sudden weakness in the person's left arm and leg - and that screams something neuro-related, maybe a stroke?\n\nBut wait, there's more. The right lower leg is swollen and tender, which is like waving a big flag for deep vein thrombosis, especially after a long flight or sitting around a lot.\n\nSo, now I'm thinking, how could a clot in the leg end up causing issues like weakness or stroke symptoms?\n\nOh, right! There's this thing called a paradoxical embolism. It can happen if there's some kind of short circuit in the heart - like a hole that shouldn't be there.\n\nLet's put this together: if a blood clot from the leg somehow travels to the

In [10]:
batch_size = 2
dataset = LlamaDataset(dataR, tokenizer=tokenizer, max_length=512)
train_loader = data.DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True, pin_memory=True)

In [11]:
a = None
for i in train_loader:
    a=i
    break

a

{'input_ids': tensor([[128000, 128256,    198,  ...,    264,   3560,   1618],
         [128000, 128256,    198,  ...,   2531,     13,    578]]),
 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1]]),
 'labels': tensor([[-100, -100, -100,  ...,  264, 3560, 1618],
         [-100, -100, -100,  ..., 2531,   13,  578]])}

In [ ]:


class LLaMAFineTuner(pl.LightningModule):
    def __init__(self, model_name, lr=2e-4):
        super().__init__()
        self.save_hyperparameters()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token

        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            load_in_4bit=True,
            device_map="auto",
            torch_dtype=torch.float16
        )

        lora_config = LoraConfig(
            r=8,
            lora_alpha=16,
            target_modules=["q_proj", "v_proj"],  # depends on model
            lora_dropout=0.05,
            bias="none",
            task_type=TaskType.CAUSAL_LM
        )

        self.model = get_peft_model(model, lora_config)

    def forward(self, input_ids, attention_mask, labels=None):
        return self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

    def training_step(self, batch, batch_idx):
        outputs = self(**batch)
        loss = outputs.loss
        self.log("train_loss", loss)
        return loss

    def configure_optimizers(self):
        return torch.optim.AdamW(self.model.parameters(), lr=self.hparams.lr)


Question
Complex_CoT
Response
